# Day 2 · 05 — Data augmentation: more pictures for free

A model with **few** training pictures starts to **memorise** them (overfitting, lesson 7): 100 % right on the pictures it has
seen, much worse on new ones.

**Data augmentation** = every time the model sees a training picture, we change it a little: flip it, shift it, make it darker…
A horse flipped left-right is still a horse, so the **label stays the same**, but the pixels are new.
The model can no longer memorise exact pixels and has to learn **what really makes a horse a horse**.

| Step | What we do |
|---|---|
| 1 | try each augmentation on one picture |
| 2 | see that the model gets a **new version every epoch** |
| 3 | good vs **bad** augmentations |
| 4 | train the same small CNN 3 times: **no** augmentation, **good** augmentation, **too strong** augmentation |
| 5 | compare the curves |

⏱️ Runs in about 5 minutes on a CPU. Uses CIFAR-10 (already downloaded for Notebook 01).
Interactive version: `Training_Visualized/13_data_augmentation.html`.

## Step 0 — Setup

We use only **5,000** training pictures on purpose: with little data, overfitting (and the effect of augmentation) is easy to see.

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

torch.manual_seed(0)
DATA_DIR = Path("..") / "data"
MEAN, STD = [0.49, 0.48, 0.45], [0.25, 0.24, 0.26]           # CIFAR-10 colour averages
LABELS = ["airplane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

train_all = datasets.CIFAR10(DATA_DIR, train=True, download=True)    # PIL pictures, no transform yet
test_all = datasets.CIFAR10(DATA_DIR, train=False, download=True)

def pick(dataset, n, seed):
    idx = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))[:n].tolist()
    return [dataset[i] for i in idx]                          # a list of (picture, label)

train_pictures = pick(train_all, 5000, 0)
test_pictures = pick(test_all, 2000, 1)

class Pictures(Dataset):
    """A list of (picture, label) plus a transform that runs EVERY time a picture is read."""
    def __init__(self, items, transform):
        self.items, self.transform = items, transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        picture, label = self.items[i]
        return self.transform(picture), label

horse = next(p for p, y in train_pictures if y == 7)
print("train:", len(train_pictures), "pictures | test:", len(test_pictures), "| one picture:", horse.size, "pixels")

## Step 1 — One picture, many augmentations

Each augmentation simulates something that really happens with photos:

In [ ]:
examples = {
    "original":               transforms.Lambda(lambda p: p),
    "flip left↔right":        transforms.RandomHorizontalFlip(p=1),
    "shift / crop":           transforms.RandomCrop(32, padding=6, padding_mode="reflect"),
    "zoom in":                transforms.RandomResizedCrop(32, scale=(0.5, 0.7)),
    "rotate ±20°":            transforms.RandomRotation(20),
    "brightness / contrast":  transforms.ColorJitter(brightness=0.6, contrast=0.6),
    "colour (hue)":           transforms.ColorJitter(saturation=0.8, hue=0.3),
    "grey":                   transforms.RandomGrayscale(p=1),
    "blur":                   transforms.GaussianBlur(5, sigma=1.2),
    "cut-out (erase a patch)": transforms.Compose([transforms.ToTensor(),
                                   transforms.RandomErasing(p=1, scale=(0.08, 0.15), value=0.5),
                                   transforms.ToPILImage()]),
}
simulates = ["", "the object facing the other way", "the object not in the centre", "the camera closer",
             "a tilted camera", "day vs evening light", "other lighting / camera colours", "old camera, no colour",
             "out of focus", "something in front of it"]

torch.manual_seed(3)
fig, axes = plt.subplots(2, 5, figsize=(15, 6.6))
for ax, (name, tf), why in zip(axes.flat, examples.items(), simulates):
    ax.imshow(tf(horse)); ax.set_title(f"{name}\n{why}", fontsize=9); ax.axis("off")
plt.suptitle("the same horse, 10 ways (the label stays 'horse')"); plt.tight_layout(); plt.show()

## Step 2 — A new version every time

We don't make copies of the pictures and save them. The transform runs **when a picture is read**, with new random numbers each time.
So in every epoch the model sees a slightly different horse. The dataset on disk stays 5,000 pictures.

This is the **good** augmentation we will train with. It is almost the same as the one in Notebook 01:

In [ ]:
normalise = [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]

no_aug = transforms.Compose(normalise)                       # test pictures ALWAYS get only this

good_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode="reflect"),   # shift up to 4 pixels
    transforms.RandomHorizontalFlip(),                              # flip half of the time
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    *normalise,
])

show = transforms.Compose(good_aug.transforms[:3])           # the same, without normalising, so we can look at it
torch.manual_seed(1)
fig, axes = plt.subplots(1, 8, figsize=(15, 2.3))
for epoch, ax in enumerate(axes, 1):
    ax.imshow(show(horse)); ax.set_title(f"epoch {epoch}", fontsize=9); ax.axis("off")
plt.suptitle("what the model sees of this ONE training picture, epoch by epoch", y=1.05); plt.show()

## Step 3 — Good and bad augmentations

**The one rule: after augmenting, the label must still be true, and the picture must still look like real data.**

| Augmentation | For CIFAR-10 photos | Why |
|---|---|---|
| flip left↔right | ✅ | a horse facing left is still a horse |
| small shift / crop | ✅ | objects are not always centred |
| a bit of brightness / colour | ✅ | light changes |
| flip **upside-down** | ❌ | test photos are never upside-down: the model wastes effort on pictures that never come |
| rotate 90° or more | ❌ | same reason |
| crop away most of the picture | ❌ | the horse may be gone, but the label still says "horse" |
| flip **digits or text** (MNIST, road signs) | ❌ | a flipped 6 is a 9, a mirrored "STOP" is not a sign |
| strong colour change when **colour is the answer** (ripe vs unripe fruit, traffic lights) | ❌ | the label depends on the colour |

Our **too strong** augmentation breaks almost all of these rules at once:

In [ ]:
too_strong = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.08, 1.0)),     # can keep only 8 % of the picture
    transforms.RandomRotation(180),                          # any angle, also upside-down
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.4),
    *normalise,
    transforms.RandomErasing(p=1, scale=(0.2, 0.4)),         # always erase a big patch
])

def unnormalise(x):
    return (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0)

torch.manual_seed(2)
fig, axes = plt.subplots(1, 8, figsize=(15, 2.3))
for ax in axes:
    ax.imshow(unnormalise(too_strong(horse))); ax.axis("off")
plt.suptitle("'too strong': is this still clearly a horse?", y=1.05); plt.show()

## Step 4 — Train the same CNN three times

Same small CNN, same 5,000 pictures, same number of epochs. **Only the training transform changes.**
The **test pictures always get `no_aug`**: we want to know how well the model does on normal, real pictures.

We record after every epoch: accuracy on the **training** pictures (as the model saw them) and on the **test** pictures.

In [ ]:
def small_cnn():
    block = lambda cin, cout: nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                                            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                                            nn.MaxPool2d(2))
    return nn.Sequential(block(3, 32), block(32, 64), block(64, 128),     # 32 → 16 → 8 → 4 pixels
                         nn.Flatten(), nn.Linear(128 * 4 * 4, 128), nn.ReLU(), nn.Linear(128, 10))

test_loader = DataLoader(Pictures(test_pictures, no_aug), batch_size=500)

@torch.inference_mode()
def accuracy(model, loader):
    model.eval()
    right = sum((model(x).argmax(1) == y).sum().item() for x, y in loader)
    return 100 * right / len(loader.dataset)

def train(name, transform, epochs=20):
    torch.manual_seed(0)
    model = small_cnn()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(Pictures(train_pictures, transform), batch_size=64, shuffle=True)
    history, start = {"train": [], "test": []}, time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train(); right = 0
        for x, y in loader:
            out = model(x); loss = loss_fn(out, y)
            opt.zero_grad(); loss.backward(); opt.step()
            right += (out.argmax(1) == y).sum().item()
        history["train"].append(100 * right / len(train_pictures))
        history["test"].append(accuracy(model, test_loader))
    print(f"{name:14s} train {history['train'][-1]:5.1f} %   test {history['test'][-1]:5.1f} %   "
          f"(best test {max(history['test']):.1f} %)   ⏱️ {time.perf_counter() - start:.0f} s")
    return history

runs = {"no augmentation": train("no augmentation", no_aug),
        "good": train("good", good_aug),
        "too strong": train("too strong", too_strong)}

## Step 5 — Compare the curves

In [ ]:
colours = {"no augmentation": "#e76f51", "good": "#2a9d8f", "too strong": "#7b2cbf"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, h) in zip(axes, runs.items()):
    epochs = range(1, len(h["train"]) + 1)
    ax.plot(epochs, h["train"], "--", color=colours[name], label="training pictures")
    ax.plot(epochs, h["test"], "-", color=colours[name], lw=2.5, label="test pictures")
    ax.fill_between(epochs, h["train"], h["test"], color=colours[name], alpha=0.12)
    ax.set_title(f"{name}: gap {h['train'][-1] - h['test'][-1]:.0f} points"); ax.set_xlabel("epoch"); ax.legend()
axes[0].set_ylabel("accuracy %"); plt.tight_layout(); plt.show()

print(f"{'':16s}{'train %':>9s}{'test %':>9s}{'gap':>7s}")
for name, h in runs.items():
    print(f"{name:16s}{h['train'][-1]:9.1f}{h['test'][-1]:9.1f}{h['train'][-1] - h['test'][-1]:7.1f}")

👀 **What to notice** (filled in after the test run)

## 🎓 What to remember

1. **Augmentation** = change training pictures randomly (flip, shift, light…) while the **label stays true**. It fights **overfitting**, especially with little data.
2. It happens **on the fly**, a new random version every epoch. No extra files, no extra labelling.
3. **Only on the training data.** Test / validation / real users get the plain transform (`Resize → ToTensor → Normalize`), exactly like Day 3's API.
4. Expect **lower training accuracy** and **higher test accuracy**: the gap shrinks. That is the goal, not a bug.
5. **Too strong** or **wrong** augmentation (upside-down, huge crops, flipped digits) hurts: the model must learn pictures that never appear.
6. It works the same for segmentation (flip the picture **and** its mask together, Notebook 06) and has versions for text and audio.

➡️ Next: `06_image_segmentation.ipynb`: a label for every pixel.